# UV IFS Extended Sources

This workbook will demonstrate the use of SYOTools with extended sources, using the "radius" property to indicate that the source is extended and the flux is a surface brightness.

## Basic SYOTools setup

In [ ]:
import numpy as np
import astropy.units as u
from syotools.models import Telescope, Source, IFS, SourceIFSExposure #models for the observatory and instruments

In [ ]:
# As of when this notebook was last updated (2026-08-24), this was the only available EAC
telescope = "EAC5"

# create the basic objects 
tel = Telescope()
# load the EAC
tel.set_from_hwome("EAC5")
# search the configured telescope for suitable bands
suitable_instruments, suitable_bands = tel.find_instrument_with("ifs")

band = "UV_IFU_Group1.UV_IFU_F1"

# this code demonstrates how to find a band with a partial name
instrument = None
for test_band in suitable_bands:
    if band in test_band:
        instrument = suitable_bands[test_band]
        break
if instrument is None:
    raise ValueError(f"Could not find an instrument with {band}")


# Make a source.
# Right now we are NOT setting the radius, so the source defaults to 0 (point source)
template = "Sbc Galaxy"
magnitude = 22
redshift = 0.0
extinction = 0.0

source = Source()
source.set_sed(template, magnitude, redshift, extinction, bandpass="galex,fuv")

tel.verbose = True

# Select the Instrument that contained that band
inst = tel.instruments[instrument]

# Make an Exposure
ifs_exp = SourceIFSExposure()
# The other way to make an exposure is to run this:
#inst.create_exposure()
ifs_exp.source = source
ifs_exp.verbose = True

inst.add_exposure(ifs_exp)
inst.band = test_band # doing it this way is a little more forgiving as an API

ifs_exp.exptime = 1 * u.hr

# this should be the last part of the setup completed, as setting the unknown unlocks auto-recalc
ifs_exp.unknown = "snr"

# The returned SNR is always a list; if you don't specify a bandpass, it will run and return all of them.
snr1 = ifs_exp.snr[0]

print("SNR:", snr1)

Let's plot the SNR curve.

In [ ]:
from matplotlib import pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(ifs_exp.wave, snr1)
ax.set_xlabel(r"Angstroms ($\AA$)")
ax.set_xlim((1000,3000))
ax.set_ylabel("SNR")

## Extended Sources

Now let's make a larger source

In [ ]:
# Now we set a radius of 0.2 arcsec - SYOTools pretends it's a uniform circle with that radius
source.set_sed(template, magnitude, redshift, extinction, radius=0.2 * u.arcsec, bandpass="galex,fuv")

# We compare that radius to the instrument's pixel scale, as read from its internal configuration dictionary:
configuration = inst.recover("configuration")
print("Pixel Scale: ", configuration["pixel_scale"])
print("Object size in pixels: ", np.pi * source.radius**2 /configuration["pixel_scale"]**2)
print("Instrument default SN extraction box: ", inst._sn_box(1750*u.AA, True))

Let's recalculate. Changing a source does not force a recalculation, so let's run one explicitly:

In [ ]:
ifs_exp.calculate_snr()

snr2 = ifs_exp.snr[0]

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(ifs_exp.wave, snr1, label="Point Source")
ax.plot(ifs_exp.wave, snr2, label="0.2\" source")
ax.legend()
ax.set_xlabel(r"Angstroms ($\AA$)")
ax.set_xlim((1000,3000))
ax.set_ylabel("SNR")